## 스마트폰 중독 감지 시스템

In [6]:
# 필수 라이브러리 설치
%pip install ultralytics opencv-python pillow matplotlib

Note: you may need to restart the kernel to use updated packages.


In [7]:
# 라이브러리 로드 및 탐지 클래스 정의

import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

# 가벼운 'n'(nano) 모델 선택 (실시간성 확보)
try:
    model = YOLO('yolov8n.pt')
    print("YOLOv8 모델 로드 성공!")
except Exception as e:
    print(print(f"모델 로드 실패: {e}\n네트워크 연결을 확인하세요."))

# COCO Dataset Class ID
PERSON_ID = 0
PHONE_ID = 67

YOLOv8 모델 로드 성공!


In [8]:
# 바운딩 박스 중심점 계산 함수

def get_box_center(bbox):
    """
    바운딩 박스좌표 [x1, y1, x2, y2]로부터 중심점 (cx, cy)를 반환합니다.
    """
    x1, y1, x2, y2 = map(int, bbox)
    cx = int((x1 + x2) / 2)
    cy = int((y1 + y2) / 2)
    return (cx, cy)

def calculate_distance(p1, p2):
    """두 점 사이의 유클리드 거리를 계산합니다."""
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def check_proximity(person_boxes, phone_boxes, max_distance=250):
    """
    검출된 모든 사람과 스마트폰 간의 거리를 전수조사하여
    임계값 이내로 인접한 스마트폰이 있는지 판별합니다.
    """
    is_using = False
    matched_pairs = []

    for p_box in person_boxes:
        p_center = get_box_center(p_box)
        for ph_box in phone_boxes:
            ph_center = get_box_center(ph_box)
            
            # 거리 측정
            dist = calculate_distance(p_center, ph_center)
            if dist < max_distance:
                is_using = True
                matched_pairs.append((p_box, ph_box)) # 시각화를 위해 매칭된 쌍 저장
                
    return is_using, matched_pairs

In [ ]:
# 실시간 추론 및 인터랙티브 UI 실행

# 웹캠 장치 열기 (0: 기본 카메라)
cap = cv2.VideoCapture(0)

# 해상도 설정 (성능과 화질의 밸런스)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("에러: 카메라를 열 수 없습니다. 다른 프로그램(Zoom, 디스코드 등)에서 카메라를 쓰고 있는지 확인하세요.")

else:
    print("스마트폰 중독 감지 시스템 가동 중... (종료하려면 웹캠 창에서 'q'를 누르세요)")

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("프레임 읽기 실패")
            break

        # YOLOv8 추론 (verbose=False로 설정하여 주피터 로그가 더러워지는 것을 방지)
        results = model(frame, verbose=False, stream=True)
        
        person_list = []
        phone_list = []

        # 결과 파싱
        for r in results:
            boxes = r.boxes
            for box in boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                
                # 신뢰도 임계값 필터링 (0.45)
                if conf > 0.45:
                    coords = box.xyxy[0].cpu().numpy()
                    if cls_id == PERSON_ID:
                        person_list.append(coords)
                    elif cls_id == PHONE_ID:
                        phone_list.append(coords)

        # 근접도 판별 로직 호출
        is_addicted, active_pairs = check_proximity(person_list, phone_list, max_distance=230)

        # ─── 시각화 단계 ───
        # 1. 모든 사람 박스 그리기 (파란색)
        for p_box in person_list:
            cv2.rectangle(frame, (int(p_box[0]), int(p_box[1])), (int(p_box[2]), int(p_box[3])), (255, 100, 0), 2)
            
        # 2. 모든 스마트폰 박스 그리기 (주황색)
        for ph_box in phone_list:
            cv2.rectangle(frame, (int(ph_box[0]), int(ph_box[1])), (int(ph_box[2]), int(ph_box[3])), (0, 120, 255), 2)

        # 3. 상태에 따른 상단 배너 및 텍스트 렌더링
        if is_addicted:
            # 경고 상태: 빨간색 배경 바 상단에 그리기
            cv2.rectangle(frame, (0, 0), (640, 50), (0, 0, 255), -1)
            # FONT_HERSHEY_DUPLEX 로 수정!
            cv2.putText(frame, "WARNING: SMARTPHONE ADDICTION DETECTED", (30, 35), 
                        cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
            
            # 사용 중인 사람과 폰 사이에 연결선(Red) 그리기
            for p_b, ph_b in active_pairs:
                p_c = get_box_center(p_b)
                ph_c = get_box_center(ph_b)
                cv2.line(frame, p_c, ph_c, (0, 0, 255), 2, cv2.LINE_AA)
        else:
            # 정상 상태: 녹색 배경 바
            cv2.rectangle(frame, (0, 0), (640, 50), (0, 150, 0), -1)
            # FONT_HERSHEY_DUPLEX 로 수정!
            cv2.putText(frame, "STATUS: CLEAR (MONITORING...)", (30, 35), 
                        cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

        # 실시간 통계 텍스트
        stats_text = f"People: {len(person_list)} | Phones: {len(phone_list)}"
        cv2.putText(frame, stats_text, (20, 460), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

        # 화면 띄우기
        cv2.imshow("VS Code YOLOv8 Detector", frame)

        # 키 입력 대기 ('q' 누르면 탈출)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# 루프 종료 후 자원 반환 (정리 단계)
cap.release()
cv2.destroyAllWindows()
print("시스템이 안전하게 종료되었습니다.")

스마트폰 중독 감지 시스템 가동 중... (종료하려면 웹캠 창에서 'q'를 누르세요)
시스템이 안전하게 종료되었습니다.
